In [19]:
import os, sys
sys.path.append('..') # to import from parent directory

from utils.prompts import render
from utils.llm_client import LLMClient # Pick LLM client
from utils.logging_utils import log_llm_call # logging ecery API call
from utils.router import pick_model, should_use_reasoning_model # general, resoning or strong
from IPython.display import Image, display, Markdown

### 01 - Zero Shot

In [3]:
promt_text, spec = render(
                            "zero_shot.v1",
                            role = "sentiment_analysis",
                            instructions = "Analyze the sentiment of the following text and respond with either 'positive', 'negative', or 'neutral'.",
                            constraints = "The sentiment should be one of the follwoing: 'positive', 'negative', or 'neutral'.",
                            format = "The sentiment is: {sentiment}"
                        )

model = pick_model("google", "general")
llm = LLMClient("google", model)

text = "I'm really happy with the service I received today! It's amazing!"
messages = [
    {
        "role": "user",
        "content": f"{promt_text}\n\n{text}"
    }
]

llm.chat(messages, temperature=0.0)

{'text': 'The sentiment is: positive',
 'usage': {'input_tokens_est': 66,
  'context_tokens_est': 0,
  'total_est': 69,
  'prompt_tokens_actual': 71,
  'completion_tokens_actual': 5,
  'total_tokens_actual': 76},
 'latency_ms': 1377,
 'raw': GenerateContentResponse(
   automatic_function_calling_history=[],
   candidates=[
     Candidate(
       content=Content(
         parts=[
           Part(
             text='The sentiment is: positive'
           ),
         ],
         role='model'
       ),
       finish_reason=<FinishReason.STOP: 'STOP'>,
       index=0
     ),
   ],
   model_version='gemini-2.5-flash',
   response_id='c1rgafHDG9Wog8UP5KqmuQ8',
   sdk_http_response=HttpResponse(
     headers=<dict len=12>
   ),
   usage_metadata=GenerateContentResponseUsageMetadata(
     candidates_token_count=5,
     prompt_token_count=71,
     prompt_tokens_details=[
       ModalityTokenCount(
         modality=<MediaModality.TEXT: 'TEXT'>,
         token_count=71
       ),
     ],
     thou

### 02 - Few Shot

In [11]:
examples = """

Example 1:
Review: I'm really happy with the product! It's bad!
Sentiment: negative
Explanation: User says product is good but he is unhappy. If user product is good or bad not his happiness prioritize that.
                    That's why we are considering the product quality as negative.

Example 2:
Review: I'm really unhappy with the product! It's amazing!
Sentiment: positive
Explanation: User says product is bad but he is happy. If user product is good or bad not his happiness prioritize that.
                    That's why we are considering the product quality as positive.

Example 3:
Review: I'm really happy with the product! It's amazing!
Sentiment: positive
Explanation: User says product is good also he is happy.

Example 4:
Review: I'm really unhappy with the product! It's bad!
Sentiment: negative
Explanation: User says product is bad also he is unhappy.

Example 5:
Review: The product is okay but it's not great.
Sentiment: neutral
Explanation: User says product is okay but not great.

Example 6:
Review: I'm not sure about the product.
Sentiment: neutral
Explanation: User is not sure about the product.
"""

promt_text, spec = render(
                            "few_shot.v1",
                            role = "sentiment_analysis", examples = examples,
                            instructions = "Analyze the sentiment of the following text and respond with either 'positive', 'negative', or 'neutral'.  Also provide the examples.",
                            constraints = "The sentiment should be one of the follwoing: 'positive', 'negative', or 'neutral'.",
                            format = "The sentiment is: {sentiment}\nExplanation: {explanation}"
                        )

model = pick_model("groq", "strong")
llm = LLMClient("groq", model)

text = "It's seems nice but It's not for me."
messages = [
    {
        "role": "user",
        "content": f"{promt_text}\n\n{text}"
    }
]

response = llm.chat(messages, temperature=0.4)
print(response['text'])

The sentiment is: neutral
Explanation: The user says the product seems nice, which indicates a positive aspect, but also states it's not for them, which is a negative aspect. The two opposing views balance each other out, resulting in a neutral sentiment.


### 03 - COT

In [21]:
model = pick_model("groq", "reason")
llm = LLMClient("groq", model)

problem = """
            A car travels 100 miles in 2 hours. What is the average speed of the car?
            Also, if the car stops for 40 minutes, what is the average speed of the car?
"""

instruction = """
                Solve the following problem step by step.
                    1. First identify whether car travelled the entire time without stopping or not.
                    2. If car stopped for x minutes and overall travelled for y, the travel duration is y-x. So the speed should be d / (y-x).
                    3. If stopping time x is mentioned, do not add it to the travel duration, because it is already included in the total travel duration.
                        So actual travel time is y (total travel time) - x (stopping time).
                    4. If car travelled the entire time without stopping, then the average speed is d / y.
                    5. If car stopped for x minutes, then the average speed is d / (y-x).
"""

promt_text, spec = render(
                        "cot_reasoning.v1",
                        role = "math_tutor",
                        problem = problem
                        )

messages = [
            {
                "role": "user",
                "content": f"""
                                text: {promt_text}
                                instruction: {instruction}
                            """
            }
]

response = llm.chat(messages, temperature=0.4)
display(Markdown(response['text']))

1. The car travels 100 miles in 2 hours without stopping.
2. The average speed without stopping is calculated as distance / time = 100 miles / 2 hours.
3. The car stops for 40 minutes, which is 40/60 = 2/3 hours. Total travel time is 2 hours, and stopping time is 2/3 hours.
4. The actual travel time when the car stops is 2 hours (total time) - 2/3 hours (stopping time) = 2 - 2/3 = 4/3 hours.
5. The average speed with stopping is calculated as distance / actual travel time = 100 miles / (2 - 2/3) hours = 100 miles / (4/3) hours.

Answer:
Average speed without stopping: 100 miles / 2 hours = 50 mph
Average speed with stopping: 100 miles / (4/3) hours = 75 mph

### 04 - TOT